In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from data.multi_frame_data import SpagDataModule
import os


root_folder="/home/tristan/data/CTC_SINETRA_seg/"
data="Fluo-N2DH-GOWT1"
model_path="/mnt/helix/FeatureNet_seg/training_logs/SiameseUNet/8feat_True_pos_1.5margin/Fluo-N2DH-GOWT1/version_0/checkpoints/best_model_epoch=165.ckpt"
subfolders=["01","02","02"]
video_path="/home/tristan/data/CTC_SINETRA_seg/Fluo-N2DH-GOWT1/02/video.tiff"
n_frames=5
data_module=SpagDataModule(
    root_folder=os.path.join(root_folder,data),
    subfolders=subfolders,
    n_frames=n_frames,
    batch_size=1,
    max_length_of_training_samples=6,
    max_length_of_validation_samples=6,
    max_length_of_testing_samples=6,
    expected_min=0,
    expected_max=255
)



In [ ]:
for batch in data_module.train_dataloader():
    images, masks = batch
    print(images.shape)
    print(masks.shape)
    

import matplotlib.pyplot as plt
plt.imshow(images.squeeze().cpu().numpy()[2], cmap='gray')
plt.show()
plt.imshow(masks.squeeze().cpu().numpy(), cmap='gray')

import napari
viewer = napari.Viewer()
viewer.add_image(images.squeeze().cpu().numpy(), name='images')
viewer.add_image(masks.squeeze().cpu().numpy(), name='masks')

In [ ]:

import os
# root_folder="/home/tristan/code/SINETRA/"
# data='hydra_flow/0.2-50.0'
# model_path="/mnt/helix/FeatureNet_seg/training_logs/SiameseUNet/32feat_True_pos/0.2-50.0/version_0/checkpoints/best_model_epoch=89.ckpt"
# subfolders=[None,None,"333"]
# video_path="/home/tristan/code/SINETRA/dataset/hydra_flow/0.2-50.0/333/video.tiff"
# n_frames=100


root_folder="/home/tristan/data/CTC_SINETRA_seg/"
data="Fluo-N2DH-GOWT1"
model_path="/mnt/helix/FeatureNet_seg/training_logs/SiameseUNet/8feat_True_pos_1.5margin/Fluo-N2DH-GOWT1/version_0/checkpoints/best_model_epoch=165.ckpt"
subfolders=["01","02","02"]
video_path="/home/tristan/data/CTC_SINETRA_seg/Fluo-N2DH-GOWT1/02/video.tiff"
n_frames=10
min_cell_size=1000
max_epochs=200

In [ ]:
from data.multi_frame_data import DefaultDataset, AugmentedDataset, _masks_folder_to_masks, SpagDataModule
import tifffile
from augmentations import VideoAugmentation


data_module=SpagDataModule(root_folder=os.path.join(root_folder, data), 
                           subfolders=subfolders, 
                           n_frames=n_frames, 
                           batch_size=1, 
             
                           augmenter=VideoAugmentation(), 
                           max_length_of_training_samples=10,
                           max_length_of_validation_samples=4,
                           max_length_of_testing_samples=4,
                           expected_min=0, 
                           expected_max=255)


In [ ]:
import napari
viewer = napari.Viewer()
viewer.add_image(tifffile.imread(video_path), name="video", colormap="gray", blending="additive")
viewer.add_labels(_masks_folder_to_masks(os.path.join(root_folder, data, subfolders[0], "tracks")), name="masks")

In [ ]:
from models.unet import SiameseUNet


model=SiameseUNet(in_channels=n_frames, spots_channels=n_frames, feat_channels=8, add_positional_encoding=True, contrastive_distance_threshold=1.5)



In [ ]:
import torch
for batch in data_module.train_dataloader():
    x,y= batch
    logits, feat_output = model(x)
    y_seg = (y > 0).to(torch.float32)
    total_loss_seg=0
    for i in range(logits.shape[1]):
        loss_seg = model.loss_fn_seg(logits[:,i:i+1], y_seg[:,i:i+1])  
        total_loss_seg += loss_seg
    total_loss_seg /= logits.shape[1]
    print(y_seg.shape, logits.shape, total_loss_seg.item())

# TESTING   

In [ ]:
import os
# root_folder="/home/tristan/code/SINETRA/"
# data='hydra_flow/0.2-50.0'
# model_path="/mnt/helix/FeatureNet_seg/training_logs/SiameseUNet/32feat_True_pos/0.2-50.0/version_0/checkpoints/best_model_epoch=89.ckpt"
# subfolders=[None,None,"333"]
# video_path="/home/tristan/code/SINETRA/dataset/hydra_flow/0.2-50.0/333/video.tiff"
# n_frames=100


root_folder="/home/tristan/data/CTC_SINETRA_seg/"
data="Fluo-N2DH-GOWT1"
model_path="/mnt/helix/FeatureNet_tra/training_logs/tracking/SiameseUNet_aug/Fluo-N2DH-GOWT1/version_2/checkpoints/best_model_epoch=372.ckpt"
subfolders=[None,None,"02"]
video_path=os.path.join(root_folder, data, subfolders[2], "video.tiff")
n_frames=5
min_cell_size=1000
max_epochs=200

In [ ]:
from data.multi_frame_data import DefaultDataset, AugmentedDataset, _masks_folder_to_masks, SpagDataModule
import tifffile
from augmentations import VideoAugmentation


data_module=SpagDataModule(root_folder=os.path.join(root_folder, data), 
                           subfolders=subfolders, 
                           n_frames=n_frames, 
                           batch_size=1, 
             
                           augmenter=VideoAugmentation(), 
                           max_length_of_training_samples=10,
                           max_length_of_validation_samples=10,
                           max_length_of_testing_samples=200,
                           expected_min=0, 
                           expected_max=255)



In [ ]:
import torch
from models.siamese_3d import SiameseUNet
# 1. Instantiate the model with the exact same arguments used during training
print(torch.cuda.is_available())
model=SiameseUNet(in_channels=1, spots_channels=1, feat_channels=8, add_positional_encoding=True, contrastive_distance_threshold=1.5, max_epochs=max_epochs)


checkpoint_path = model_path  # Path to your checkpoint file

# 2. Load the raw checkpoint file
checkpoint = torch.load(checkpoint_path, map_location="cpu")

# 3. Handle different save formats
if "state_dict" in checkpoint:
    # Sometimes raw saves still wrap the weights in a 'state_dict' key
    model.load_state_dict(checkpoint["state_dict"])
    print("Loaded model from checkpoint with 'state_dict' key.")
else:
    # Otherwise, the checkpoint is likely just the weights dictionary itself
    model.load_state_dict(checkpoint)
    # =print("Loaded model from checkpoint without 'state_dict' key.")
model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
model.eval()



In [ ]:
# from models.unet import SiameseUNet

# model=SiameseUNet.load_from_checkpoint(model_path)

In [ ]:
torch.cuda.empty_cache
preds=model.predict_from_dataloader(data_module.test_dataloader(), keep_pseudo_probs=True)
import pickle as pkl
#create directory if it doesn't exist
os.makedirs(f"./results/{data}/{n_frames}_frames", exist_ok=True)
pkl.dump(preds, open(f"./results/{data}/{n_frames}_frames/3d_preds_v2_best.pkl", "wb"))

In [ ]:
print(preds[0][0].shape)

In [ ]:
import numpy as np
masks=[]
gts=[]
all_pseudo_probs=[]
for i in range(len(preds)):
    for j in range(n_frames-4):
        pseudo_probs = preds[i][0][0,j].squeeze().detach().cpu().numpy()
        all_pseudo_probs.append(pseudo_probs)
        masks.append((pseudo_probs>0.5).astype(np.uint16))

        gts.append(data_module.test_ds[i][1][j].squeeze().detach().cpu().numpy().astype(np.uint16))

masks=np.array(masks)
gts=np.array(gts)
all_pseudo_probs=np.array(all_pseudo_probs)
print(masks.shape)
print(gts.shape)
import napari
viewer = napari.Viewer()
viewer.add_image(tifffile.imread(video_path), name="video", colormap="gray", blending="additive")
viewer.add_labels( masks, name="predicted masks")
viewer.add_image(all_pseudo_probs, name="pseudo probabilities", colormap="viridis")
viewer.add_labels(gts, name="ground truth masks")

# DENSITY

In [ ]:
from sklearn.cluster import MeanShift
from dbscan import DBSCAN
from hdbscan import HDBSCAN
import numpy as np
import torch
def cluster_features_to_instances(dense_features, foreground_mask, method='dbscan', margin=1.5, min_cell_size=100):
    """
    Clusters 64-d pixel embeddings into discrete instance masks.
    
    Args:
        dense_features: Tensor or numpy array of shape (C, H, W)
        foreground_mask: Binary mask of shape (H, W) where 1=cell, 0=background.
        method: 'dbscan' (recommended) or 'meanshift'.
        margin: The distance margin used during your contrastive training.
        min_cell_size: The minimum size of a cell in pixels.
        
    Returns:
        instance_mask: Numpy array of shape (H, W) with unique integer IDs for cells.
    """
    # 1. Convert to numpy
    if isinstance(dense_features, torch.Tensor):
        dense_features = dense_features.detach().cpu().squeeze(0).numpy()
    if isinstance(foreground_mask, torch.Tensor):
        foreground_mask = foreground_mask.detach().cpu().squeeze(0).numpy()


    T, C, H, W = dense_features.shape
    print(f"Clustering features of shape: {dense_features.shape} with foreground mask shape: {foreground_mask.shape}")
    # 2. Flatten features: (H*W, C)
    features_flat = np.moveaxis(dense_features, 1, -1).reshape(-1, C)
    fg_mask_flat = foreground_mask.flatten() > 0
    
    # 3. Extract only foreground pixels to cluster
    # This drastically speeds up clustering and prevents background noise
    fg_features = features_flat[fg_mask_flat]
    
    # Create an empty instance mask (0 = background)
    instance_mask_flat = np.zeros(T*H * W, dtype=np.int32)
    
    # Edge case: No foreground detected
    if len(fg_features) == 0:
        return instance_mask_flat.reshape(T, H, W)

    labels = None  # Initialize labels variable
    # 4. Configure the Clustering Algorithm
    if method == 'dbscan':
        # eps should be slightly less than your training margin. 
        # If your margin was 1.5, points within the same cluster should be 
        # very close (e.g., < 0.5), while different clusters are > 1.5 aparigrapht.
        eps_threshold = margin * 0.2
        
        # min_samples prevents tiny noise dots from becoming cells.
        # Set this based on your minimum expected cell size (in pixels).
        #SKLEARN
        # clusterer = DBSCAN(eps=eps_threshold, min_samples=min_cell_size*T, metric='euclidean')
        #DBSCAN
        print(fg_features.shape)
        labels, core_samples = DBSCAN(fg_features, eps=eps_threshold, min_samples=min_cell_size)
    elif method == 'meanshift':
        # MeanShift bandwidth is also tied to your margin
        clusterer = MeanShift(bandwidth=margin * 0.5, bin_seeding=True)
    
    elif method == 'hdbscan':
        # HDBSCAN is a density-based clustering algorithm that can handle variable density clusters.
        # The min_cluster_size parameter controls the smallest size of clusters to consider.
        clusterer = HDBSCAN(min_cluster_size=min_cell_size*T, metric='euclidean')
        
    elif method == 'leiden':
        import leidenalg as la
        import igraph as ig
        # Leiden is a graph-based clustering algorithm. It requires constructing a k-nearest neighbor graph.
        # This is more complex and may require additional preprocessing.
        raise NotImplementedError("Leiden clustering is not yet implemented in this function.")
        
    else:
        raise ValueError("Method must be 'dbscan' or 'meanshift'")

    # 5. Run Clustering
    print(f"Clustering {len(fg_features)} foreground pixels using {method.upper()}...")
    if labels is None:
        labels = clusterer.fit_predict(fg_features)
    
    # DBSCAN returns -1 for noise/outliers. We want to shift our IDs so they 
    # start at 1, keeping 0 exclusively for the background.
    if method == 'dbscan':
        # Map noise (-1) to background (0), and shift other clusters up by 1
        labels = np.where(labels == -1, -1, labels) + 1
    else:
        # MeanShift returns labels 0, 1, 2... Shift up by 1.
        labels = labels + 1

    # 6. Reconstruct the 2D Instance Mask
    # Place the cluster labels back into the original spatial locations
    instance_mask_flat[fg_mask_flat] = labels
    instance_mask_2d = instance_mask_flat.reshape(T, H, W)
    print(f"Detected {len(np.unique(instance_mask_2d)) - 1} instances.")
    return instance_mask_2d




In [ ]:
import pickle as pkl
import numpy as np
preds=pkl.load(open("./results/Fluo-N2DH-GOWT1/1_frames/preds_2d_seg.pkl", "rb"))
foreground_mask, dense_features = preds[0][0] > 0.5, preds[0][1]
for i in range(1,2):
    foreground_mask = np.concatenate((foreground_mask, preds[i][0] > 0.5), axis=0)
    dense_features = np.concatenate((dense_features, preds[i][1]), axis=0)
print(foreground_mask.shape, dense_features.shape)

instance_mask = cluster_features_to_instances(dense_features, foreground_mask > 0.5, method='dbscan', margin=1.5, min_cell_size=1000)


In [ ]:
import napari
viewer = napari.Viewer()
import tifffile
video_path="/home/tristan/data/CTC_SINETRA_seg/Fluo-N2DH-GOWT1/02/video.tiff"
viewer.add_image(tifffile.imread(video_path)[2:5], name="video")
viewer.add_labels(instance_mask, name="instance mask")

# GRAPH

In [ ]:
from scipy.optimize import linear_sum_assignment
import numpy as np
import networkx as nx
from networkx.algorithms import community

import numpy as np
import torch
import networkx as nx

def build_spatial_feature_graph_nx(
    dense_features, 
    binary_mask, 
    spatial_connectivity=8, 
    affinity_sigma=1.0,
    distance_threshold=0.3
):
    """
    Constructs a 2D spatial-adjacency graph for a single frame using NetworkX.
    
    Args:
        dense_features: Tensor or array of shape (C, H, W)
        binary_mask: Tensor or array of shape (H, W) where 1=target pixel
        spatial_connectivity: 4 or 8 for 2D spatial connectivity
        affinity_sigma: Bandwidth for RBF kernel to convert distance to similarity
        distance_threshold: Edges with distance greater than this are culled 
                            before graph construction to save memory.
                            
    Returns:
        g: networkx.Graph object
        node_grid: (H, W) mapping from spatial coordinates to node IDs
    """
    # 1. Ensure inputs are NumPy arrays
    if isinstance(dense_features, torch.Tensor):
        dense_features = dense_features.detach().cpu().numpy()
    if isinstance(binary_mask, torch.Tensor):
        binary_mask = binary_mask.detach().cpu().numpy()

    if dense_features.ndim != 3 or binary_mask.ndim != 2:
        raise ValueError(f"Expected (C, H, W) and (H, W), got {dense_features.shape} and {binary_mask.shape}")

    C, H, W = dense_features.shape
    fg_bool = binary_mask > 0

    # 2. Map target pixels to node IDs (0 to N-1)
    node_grid = np.full((H, W), -1, dtype=np.int32)
    node_indices = np.argwhere(fg_bool) # (N, 2) -> [y, x]
    N = len(node_indices)

    # Fast-fail if no foreground pixels exist
    if N == 0:
        return nx.Graph(), node_grid

    node_grid[fg_bool] = np.arange(N, dtype=np.int32)

    # Reshape features from (C, H, W) -> (H, W, C) -> extract (N, C)
    fg_features = np.moveaxis(dense_features, 0, -1)[fg_bool].astype(np.float32)

    # 3. Define 2D directional neighbor offsets
    offsets = [(0, 1), (1, 0)] # Down, Right
    if spatial_connectivity == 8:
        offsets.extend([(1, 1), (1, -1)]) # Diagonals: Down-Right, Down-Left

    # 4. Vectorized spatial edge extraction
    src_nodes_list, dst_nodes_list = [], []

    for dy, dx in offsets:
        y_src = slice(max(0, -dy), min(H, H - dy))
        x_src = slice(max(0, -dx), min(W, W - dx))
        
        y_dst = slice(max(0, dy), min(H, H + dy))
        x_dst = slice(max(0, dx), min(W, W + dx))
        
        src_ids = node_grid[y_src, x_src]
        dst_ids = node_grid[y_dst, x_dst]
        
        # Keep only pairs where both source and dest are valid foreground pixels
        valid = (src_ids >= 0) & (dst_ids >= 0)
        
        src_nodes_list.append(src_ids[valid])
        dst_nodes_list.append(dst_ids[valid])

    if len(src_nodes_list) == 0:
        # Edge case: No adjacent pixels
        g = nx.Graph()
        g.add_nodes_from(range(N))
        return g, node_grid

    src_indices = np.concatenate(src_nodes_list)
    dst_indices = np.concatenate(dst_nodes_list)

    # 5. Calculate Feature Distances
    diffs = fg_features[src_indices] - fg_features[dst_indices]
    distances = np.linalg.norm(diffs, axis=1)
    
    # 6. Apply pre-filtering (Massive optimization for NetworkX)
    if distance_threshold is not None:
        valid_mask = distances <= distance_threshold
        src_indices = src_indices[valid_mask]
        dst_indices = dst_indices[valid_mask]
        distances = distances[valid_mask]
        
    affinities = np.exp(-distances / (affinity_sigma + 1e-8))

    # 7. Build the NetworkX graph in bulk
    g = nx.Graph()
    g.add_nodes_from(range(N))
    
    # NetworkX expects edges as (u, v, attribute_dict)
    # Using a list comprehension is the fastest way to package this for NetworkX
    edge_attributes = [{'distance': d, 'weight': w} for d, w in zip(distances.tolist(), affinities.tolist())]
    edges_with_attrs = zip(src_indices.tolist(), dst_indices.tolist(), edge_attributes)
    
    g.add_edges_from(edges_with_attrs)
    
    return g, node_grid

def cluster_spatial_graph(g, node_grid, binary_mask, min_cell_size=50, resolution=0.0000001):
    """
    Step 2: Uses Louvain to cluster the spatial graph into individual cells.
    """
    if g.number_of_nodes() == 0:
        return np.zeros_like(binary_mask, dtype=np.int32)
        
    communities = community.louvain_communities(g, weight='weight', resolution=resolution)
    
    node_to_cluster_map = np.zeros(g.number_of_nodes(), dtype=np.int32)
    cluster_id = 1
    
    for node_set in communities:
        if len(node_set) >= min_cell_size:
            node_to_cluster_map[list(node_set)] = cluster_id
            cluster_id += 1
            
    instance_mask = np.zeros_like(binary_mask, dtype=np.int32)
    valid_pixels = node_grid >= 0
    instance_mask[valid_pixels] = node_to_cluster_map[node_grid[valid_pixels]]
    
    return instance_mask


import numpy as np
import networkx as nx

def build_temporal_graph(features_video, instance_masks_video, max_temporal_distance=1.5):
    """
    Step 3: Builds a directed graph linking instances across consecutive frames 
    based on their average 64-D feature vector.
    """
    # 1. Convert to NumPy to fix indexing quirks and prevent NetworkX memory leaks
    if hasattr(features_video, 'detach'):
        features_video = features_video.detach().cpu().numpy()
        
    T = features_video.shape[0]
    g_temp = nx.DiGraph()
    
    print("Computing mean feature embeddings for all detected cells...")
    for t in range(T):
        instances = np.unique(instance_masks_video[t])
        instances = instances[instances > 0] # Ignore background
        
        # Extract the frame FIRST. 
        # This guarantees frame_features is strictly (C, H, W)
        frame_features = features_video[t]
        frame_mask = instance_masks_video[t]
        
        for inst_id in instances:
            # frame_features[:, boolean_mask] safely guarantees shape is (C, N)
            cell_pixels = frame_features[:, frame_mask == inst_id]
            
            # Average across the N pixels (axis 1) to get a strict (C,) 64-D signature
            mean_signature = cell_pixels.mean(axis=1) 
            
            # Node name is a tuple: (frame_index, cell_id)
            g_temp.add_node((t, inst_id), feature=mean_signature)
            
    print("Connecting temporal edges...")
    for t in range(T - 1):
        nodes_t = [n for n in g_temp.nodes if n[0] == t]
        nodes_next = [n for n in g_temp.nodes if n[0] == t + 1]
        
        for u in nodes_t:
            feat_u = g_temp.nodes[u]['feature']
            for v in nodes_next:
                feat_v = g_temp.nodes[v]['feature']
                dist = np.linalg.norm(feat_u - feat_v)
                
                if dist <= max_temporal_distance:
                    g_temp.add_edge(u, v, distance=dist)
                    
    return g_temp

def cluster_temporal_graph(temporal_graph, instance_masks_video):
    """
    Step 4: Enforces 1-to-1 tracking constraints. Resolves the graph into globally 
    consistent cell IDs across the entire video.
    """
    T = instance_masks_video.shape[0]
    tracked_video = np.zeros_like(instance_masks_video, dtype=np.int32)
    
    global_id_counter = 1
    mapping = {} # Maps original (t, local_id) to the new global Track ID
    
    print("Resolving 1-to-1 tracking constraints...")
    
    # Initialize Frame 0 with brand new Global IDs
    nodes_0 = [n for n in temporal_graph.nodes if n[0] == 0]
    for n in nodes_0:
        mapping[n] = global_id_counter
        tracked_video[0][instance_masks_video[0] == n[1]] = global_id_counter
        global_id_counter += 1
        
    # Process bipartite matching frame by frame
    for t in range(T - 1):
        nodes_t = [n for n in temporal_graph.nodes if n[0] == t]
        nodes_next = [n for n in temporal_graph.nodes if n[0] == t + 1]
        
        if not nodes_t or not nodes_next:
            continue
            
        # Build the Cost Matrix for the Hungarian Algorithm
        cost_matrix = np.full((len(nodes_t), len(nodes_next)), np.inf)
        
        for i, u in enumerate(nodes_t):
            for j, v in enumerate(nodes_next):
                if temporal_graph.has_edge(u, v):
                    cost_matrix[i, j] = temporal_graph[u][v]['distance']
                    
        # linear_sum_assignment finds the mathematically optimal 1-to-1 
        # that minimizes the total distance across all cells
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        
        assigned_next_nodes = set()
        
        # Carry over the global IDs for matched cells
        for i, j in zip(row_ind, col_ind):
            if cost_matrix[i, j] < np.inf:
                u = nodes_t[i]
                v = nodes_next[j]
                
                global_track_id = mapping[u]
                mapping[v] = global_track_id
                tracked_video[t+1][instance_masks_video[t+1] == v[1]] = global_track_id
                
                assigned_next_nodes.add(v)
                
        # Assign brand new Global IDs to any cells in t+1 that didn't get matched
        # (These are new cells entering the frame or dividing)
        for v in nodes_next:
            if v not in assigned_next_nodes:
                mapping[v] = global_id_counter
                tracked_video[t+1][instance_masks_video[t+1] == v[1]] = global_id_counter
                global_id_counter += 1

    print(f"Tracking complete. Found {global_id_counter - 1} unique global tracks.")
    return tracked_video

In [ ]:
import pickle as pkl
import numpy as np
from tqdm import tqdm
preds=pkl.load(open("./results/Fluo-N2DH-GOWT1/5_frames/3d_preds_v1_best.pkl", "rb"))
foreground_mask, dense_features = preds[0][0].squeeze(0) > 0.5, preds[0][1].squeeze(0)
for i in range(1,10):
    foreground_mask = np.concatenate((foreground_mask, preds[i][0].squeeze(0) > 0.5), axis=0)
    dense_features = np.concatenate((dense_features, preds[i][1].squeeze(0)), axis=0)
print(foreground_mask.shape, dense_features.shape)
# foreground_mask = foreground_mask[:,1,:,:].unsqueeze(1)
# dense_features =], name="video") dense_features[:,1,:,:].unsqueeze(1)
print(dense_features.shape, foreground_mask.shape)

# Assuming you have:
# features_3d shape (T, C, H, W)
# mask_3d shape (T, H, W)
T = dense_features.shape[0]

# 1. Process Spatial Data Frame-by-Frame
raw_instance_masks = np.zeros_like(foreground_mask, dtype=np.int32)

for t in tqdm(range(T), desc=f"Processing {T} frames"):
    g_spatial, node_grid = build_spatial_feature_graph_nx(
        dense_features[t], 
        foreground_mask[t], 
        distance_threshold=1.5)
    print(f"Frame {t}: Graph has {g_spatial.number_of_nodes()} nodes and {g_spatial.number_of_edges()} edges.")
    raw_instance_masks[t] = cluster_spatial_graph(
        g_spatial, 
        node_grid, 
        foreground_mask[t],
        min_cell_size=5,
        resolution=1
    )
    print(f"Frame {t}: Detected {len(np.unique(raw_instance_masks[t])) - 1} instances.")
# 2. Process Temporal Data Globally
g_temporal = build_temporal_graph(dense_features, raw_instance_masks, max_temporal_distance=1.5)

# 3. Final Output: A perfect (T, H, W) array where cell #5 keeps ID 5 across all frames
tracked_video_masks = cluster_temporal_graph(g_temporal, raw_instance_masks)

In [ ]:
import napari
viewer = napari.Viewer()
import tifffile
video_path="/home/tristan/data/CTC_SINETRA_seg/Fluo-N2DH-GOWT1/02/video.tiff"
viewer.add_image(tifffile.imread(video_path)[2:5], name="video")
viewer.add_labels(raw_instance_masks, name="tracked instance masks")

# CONSTRAINED DBSCAN FOR TRACKING


In [ ]:
import pickle as pkl
import numpy as np
from tqdm import tqdm

n_tested=80
preds=pkl.load(open("./results/Fluo-N2DH-GOWT1/5_frames/3d_preds_v2_best.pkl", "rb"))
foreground_mask, dense_features = preds[0][0].squeeze(0) > 0.5, preds[0][1].squeeze(0)
for i in range(1,n_tested):
    foreground_mask = np.concatenate((foreground_mask, preds[i][0].squeeze(0) > 0.5), axis=0)
    dense_features = np.concatenate((dense_features, preds[i][1].squeeze(0)), axis=0)
print(foreground_mask.shape, dense_features.shape)

In [ ]:
#graph based segmentation
import numpy as np
from tracking_functions import build_spatial_feature_graph_nx, cluster_spatial_graph
instance_masks = np.zeros_like(foreground_mask, dtype=np.int32)
for i in range(n_tested):
    g_spatial, node_grid = build_spatial_feature_graph_nx(
        dense_features[i], 
        foreground_mask[i], 
        distance_threshold=1.5)
    print(f"Frame {i}: Graph has {g_spatial.number_of_nodes()} nodes and {g_spatial.number_of_edges()} edges.")
    instance_mask = cluster_spatial_graph(
        g_spatial, 
        node_grid, 
        foreground_mask[i],
        min_cell_size=1000,
        resolution=0.0000001
    )
    print(f"Frame {i}: Detected {len(np.unique(instance_mask)) - 1} instances.")
    instance_masks[i] = instance_mask


In [ ]:
import napari 
viewer = napari.Viewer()
import tifffile
video_path="/home/tristan/data/CTC_SINETRA_seg/Fluo-N2DH-GOWT1/02/video.tiff"
viewer.add_image(tifffile.imread(video_path)[2:n_tested+2], name="video")
viewer.add_labels(instance_masks, name="instance masks")

In [ ]:
from scipy.ndimage import center_of_mass
def instance_mask_to_features(instance_mask, dense_features):
    """
    Converts an instance mask and dense features into a feature vector for each instance.
    
    Args:
        instance_mask: 2D array of shape (H, W) with unique integer IDs for instances.
        dense_features: 3D array of shape (C, H, W) representing dense features.
    Returns:
        instance_features: (N, C) array of averaged 64-d cell embeddings.
        loc: (N, 2) array of cell centroids in pixel coordinates.
    """
    unique_ids = np.unique(instance_mask)
    unique_ids = unique_ids[unique_ids > 0]  # Exclude background (0)
    
    instance_features = []
    loc = []
    
    for inst_id in unique_ids:
        mask = instance_mask == inst_id
        if np.sum(mask) == 0:
            continue
        # Average the dense features over the pixels belonging to this instance
        avg_feature = dense_features[:, mask].mean(axis=1)
        instance_features.append(avg_feature)
        # Calculate the centroid of the instance
        centroid = center_of_mass(mask)
        loc.append(centroid)
    
    return np.array(instance_features), np.array(loc)

instance_features=np.empty((0, dense_features.shape[1]), dtype=np.float32)
frames=[]
inst_locs=np.empty((0, 2), dtype=np.float32)
for i in range(n_tested):
    inst_features, inst_loc = instance_mask_to_features(instance_masks[i], dense_features[i])
    instance_features= np.concatenate((instance_features, inst_features), axis=0)
    frames= np.concatenate((frames, np.ones(inst_features.shape[0])*i), axis=0)
    inst_locs= np.concatenate((inst_locs, inst_loc), axis=0)
print(instance_features.shape, frames.shape, inst_locs.shape)

In [ ]:
# visualize the instance features in 2D using Tsne
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=42)
instance_features_2d = tsne.fit_transform(instance_features)   
from matplotlib import pyplot as plt
plt.figure(figsize=(8, 6))
plt.scatter(instance_features_2d[:, 0], instance_features_2d[:, 1], c=frames, cmap='viridis', s=5)
plt.colorbar(label='Frame Index')
plt.title('t-SNE of Instance Features Colored by Frame Index')

In [ ]:


import numpy as np
from scipy.spatial.distance import cdist

def constrained_dbscan_tracking(features, frames, eps=0.5, min_samples=2):
    """
    Custom DBSCAN that enforces a Cannot-Link constraint:
    Two points cannot belong to the same cluster if they share the same frame ID.
    
    Args:
        features: (N, C) array of averaged 64-d cell embeddings.
        frames: (N,) integer array of the frame ID for each cell.
        eps: Maximum feature distance to be considered the same track.
        min_samples: Minimum track length (in frames) to be kept.
        
    Returns:
        labels: (N,) array of Track IDs. -1 indicates noise/untracked.
    """
    N = len(features)
    labels = np.full(N, -1, dtype=np.int32)
    visited = np.zeros(N, dtype=bool)
    
    # Precompute all pairwise distances (Fast for a few thousand cells)
    dist_matrix = cdist(features, features, metric='euclidean')
    
    cluster_id = 1
    
    for p_idx in range(N):
        if visited[p_idx]:
            continue
            
        visited[p_idx] = True
        
        # 1. Region Query: Find all points within eps distance
        neighbors = np.where(dist_matrix[p_idx] <= eps)[0]
        
        # 2. Hard Constraint: Remove neighbors in the exact same frame as the seed
        neighbors = [n for n in neighbors if frames[n] != frames[p_idx]]
        
        # Check density (Core point)
        if len(neighbors) + 1 < min_samples:
            continue # Remains -1 (Noise)
            
        # 3. Initialize new track
        labels[p_idx] = cluster_id
        frames_in_cluster = {frames[p_idx]}
        
        # Sort neighbors by distance to the seed point! 
        # This ensures the closest feature match claims the frame slot first.
        neighbors = sorted(neighbors, key=lambda n: dist_matrix[p_idx, n])
        queue = list(neighbors)
        
        # 4. Expand Cluster
        while queue:
            q_idx = queue.pop(0)
            
            if not visited[q_idx]:
                visited[q_idx] = True
                q_neighbors = np.where(dist_matrix[q_idx] <= eps)[0]
                q_neighbors = [n for n in q_neighbors if frames[n] != frames[q_idx]]
                
                if len(q_neighbors) + 1 >= min_samples:
                    # Sort expansion candidates by distance to q_idx
                    q_neighbors = sorted(q_neighbors, key=lambda n: dist_matrix[q_idx, n])
                    queue.extend(q_neighbors)
                    
            # 5. Apply Cannot-Link Constraint during assignment
            if labels[q_idx] == -1:
                if frames[q_idx] not in frames_in_cluster:
                    labels[q_idx] = cluster_id
                    frames_in_cluster.add(frames[q_idx])
                    
        cluster_id += 1
        
    return labels

In [ ]:
labels = constrained_dbscan_tracking(instance_features, frames, eps=1.5, min_samples=2)
print(f"Found {len(np.unique(labels)) - 1} unique tracks (excluding noise).")
print(f"Track ID distribution: {np.unique(labels, return_counts=False)}")


In [ ]:
print(labels.shape)

In [ ]:
napari_tracks= np.concatenate([labels[:, np.newaxis],frames[:, np.newaxis], inst_locs.astype(np.int32)], axis=1).astype(np.int32)
napari_tracks_filtered=napari_tracks[napari_tracks[:,0]!=-1]
print(napari_tracks.shape)
print(napari_tracks_filtered)


In [ ]:
import napari 
viewer = napari.Viewer()
import tifffile
video_path="/home/tristan/data/CTC_SINETRA_seg/Fluo-N2DH-GOWT1/02/video.tiff"
viewer.add_image(tifffile.imread(video_path)[2:n_tested+2], name="video")
viewer.add_labels(instance_masks, name="instance masks")
viewer.add_tracks(napari_tracks_filtered, tail_width=5, name='my_tracks')

In [ ]:
import napari
import tifffile
from tracking_functions import build_spatial_feature_graph_nx, cluster_spatial_graph, instance_mask_to_features, constrained_dbscan_tracking, dbscan_with_connected_components
import pickle as pkl
import numpy as np
from tqdm import tqdm

n_tested=80


preds=pkl.load(open("./results/Fluo-N2DH-GOWT1/1_frames/preds_2d_seg.pkl", "rb"))

foreground_mask, dense_features = preds[0][0] > 0.5, preds[0][1]
print(foreground_mask.shape, dense_features.shape)
for i in range(1,n_tested):
    foreground_mask = np.concatenate((foreground_mask, preds[i][0] > 0.5), axis=0)
    dense_features = np.concatenate((dense_features, preds[i][1]), axis=0)
print(foreground_mask.shape, dense_features.shape)



#graph based segmentation
# instance_masks = np.zeros_like(foreground_mask, dtype=np.int32)
# for i in range(n_tested):
#     g_spatial, node_grid = build_spatial_feature_graph_nx(
#         dense_features[i], 
#         foreground_mask[i], 
#         distance_threshold=1.5)
#     print(f"Frame {i}: Graph has {g_spatial.number_of_nodes()} nodes and {g_spatial.number_of_edges()} edges.")
#     instance_mask = cluster_spatial_graph(
#         g_spatial, 
#         node_grid, 
#         foreground_mask[i],
#         min_cell_size=1000,
#         resolution=0.0000001
#     )
#     print(f"Frame {i}: Detected {len(np.unique(instance_mask)) - 1} instances.")
#     instance_masks[i] = instance_mask


#density based segmentation
instance_masks = np.zeros_like(foreground_mask, dtype=np.int32)
for i in range(n_tested):
    instance_mask = dbscan_with_connected_components(
        dense_features[i], 
        foreground_mask[i],
        eps=50,
        min_samples=1000
    )
    print(instance_mask.shape)
    print(f"Frame {i}: Detected {len(np.unique(instance_mask)) - 1} instances.")
    print(f"Frame {i}: Unique instance IDs: {np.unique(instance_mask)}")
    instance_masks[i] = instance_mask

instance_features=np.empty((0, dense_features.shape[1]), dtype=np.float32)
frames=[]
inst_locs=np.empty((0, 2), dtype=np.float32)
for i in range(n_tested):
    inst_features, inst_loc = instance_mask_to_features(instance_masks[i], dense_features[i])
    instance_features= np.concatenate((instance_features, inst_features), axis=0)
    frames= np.concatenate((frames, np.ones(inst_features.shape[0])*i), axis=0)
    inst_locs= np.concatenate((inst_locs, inst_loc), axis=0)
    

# contrained DBSCAN tracking
labels = constrained_dbscan_tracking(instance_features, frames, eps=0.2, min_samples=2)

napari_tracks= np.concatenate([labels[:, np.newaxis],frames[:, np.newaxis], inst_locs.astype(np.int32)], axis=1).astype(np.int32)
napari_tracks_filtered=napari_tracks[napari_tracks[:,0]!=-1]
print(f"Found {len(np.unique(labels)) - 1} unique tracks (excluding noise).")
print(f"Track ID distribution: {np.unique(labels, return_counts=True)}")

viewer = napari.Viewer()
video_path="/home/tristan/data/CTC_SINETRA_seg/Fluo-N2DH-GOWT1/02/video.tiff"
viewer.add_image(tifffile.imread(video_path)[2:n_tested+2], name="video")
viewer.add_labels(instance_masks, name="instance masks")
viewer.add_tracks(napari_tracks_filtered)

In [ ]:
# visualize the instance features in 2D using Tsne
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=42)
instance_features_2d = tsne.fit_transform(instance_features)   
from matplotlib import pyplot as plt
plt.figure(figsize=(8, 6))
plt.scatter(instance_features_2d[:, 0], instance_features_2d[:, 1], c=frames, cmap='viridis', s=5)
plt.colorbar(label='Frame Index')
plt.title('t-SNE of Instance Features Colored by Frame Index')

# KALMAN ATTEMPT

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

class FeatureTrack:
    """
    A 1st-order Kalman Filter designed specifically for high-dimensional feature spaces.
    Tracks both the Feature Vector (C,) and its Rate of Change/Velocity (C,).
    """
    def __init__(self, initial_feature, track_id):
        self.track_id = track_id
        C = len(initial_feature)
        
        # State Vector X: [features (C,), velocities (C,)]
        self.X = np.zeros((2 * C, 1), dtype=np.float32)
        self.X[:C, 0] = initial_feature
        
        # Transition Matrix F (Constant Velocity Model)
        self.F = np.eye(2 * C, dtype=np.float32)
        self.F[:C, C:] = np.eye(C, dtype=np.float32)
        
        # Measurement Matrix H (We only observe the features, not the velocity)
        self.H = np.zeros((C, 2 * C), dtype=np.float32)
        self.H[:, :C] = np.eye(C, dtype=np.float32)
        
        # Covariance Matrices
        self.P = np.eye(2 * C, dtype=np.float32) * 10.0
        self.Q = np.eye(2 * C, dtype=np.float32) * 0.01 
        self.R = np.eye(C, dtype=np.float32) * 0.1
        
        # Lifecycle Management
        self.time_since_update = 0
        self.hits = 1
        self.active = True 
        
    def predict(self):
        self.X = np.dot(self.F, self.X)
        self.P = np.dot(np.dot(self.F, self.P), self.F.T) + self.Q
        self.time_since_update += 1
        return self.X[:len(self.X)//2, 0] 

    def update(self, measurement):
        Z = measurement.reshape(-1, 1)
        Y = Z - np.dot(self.H, self.X)
        S = np.dot(np.dot(self.H, self.P), self.H.T) + self.R
        K = np.dot(np.dot(self.P, self.H.T), np.linalg.inv(S))
        self.X = self.X + np.dot(K, Y)
        I = np.eye(self.P.shape[0])
        self.P = np.dot((I - np.dot(K, self.H)), self.P)
        self.time_since_update = 0
        self.hits += 1



In [ ]:
def kalman_feature_tracking(features, frames, max_feature_distance=1.0, max_missed_frames=3, min_hits=3):
    """
    Predictive tracking in high-dimensional space using a Kalman Filter.
    
    Args:
        features: (N, C) numpy array of averaged cell embeddings.
        frames: (N,) integer array of the frame ID for each cell.
        max_feature_distance: Cost threshold to reject a match (controls birth/death).
        max_missed_frames: Number of frames a track can go unseen before dying.
        min_hits: Minimum number of frames a track must exist to not be labeled as noise (-1).
        
    Returns:
        labels: (N,) integer array of Track IDs. -1 indicates noise/untracked.
    """
    N = len(features)
    
    # Initialize all cells as noise (-1)
    labels = np.full(N, -1, dtype=np.int32)
    
    # Ensure we iterate frames chronologically
    unique_frames = np.sort(np.unique(frames))
    
    active_tracks = [] # Currently alive tracks
    all_tracks = []    # History of all tracks ever created
    
    # Maps track_id -> list of indices in the flat 'features' array
    track_history = {} 
    global_track_id_counter = 1
    
    for t in unique_frames:
        # Find all cells belonging to the current frame
        curr_indices = np.where(frames == t)[0]
        curr_features = features[curr_indices]
        
        # 1. Predict next feature state for all active tracks
        predicted_features = [tr.predict() for tr in active_tracks]
        
        # 2. Match predictions to actual frame detections
        if len(active_tracks) > 0 and len(curr_indices) > 0:
            cost_matrix = np.zeros((len(active_tracks), len(curr_indices)))
            
            for i, tr_pred in enumerate(predicted_features):
                for j, det_feat in enumerate(curr_features):
                    cost_matrix[i, j] = np.linalg.norm(tr_pred - det_feat)
                    
            row_ind, col_ind = linear_sum_assignment(cost_matrix)
        else:
            row_ind, col_ind = [], []
            cost_matrix = np.empty((0, 0))
            
        matched_track_indices = set()
        matched_det_indices = set()
        
        # 3. Update matched tracks
        for i, j in zip(row_ind, col_ind):
            if cost_matrix[i, j] <= max_feature_distance:
                track = active_tracks[i]
                det_feat = curr_features[j]
                global_idx = curr_indices[j]
                
                track.update(det_feat)
                track_history[track.track_id].append(global_idx)
                
                matched_track_indices.add(i)
                matched_det_indices.add(j)
                
        # 4. Birth: Handle Unmatched Detections
        for j, global_idx in enumerate(curr_indices):
            if j not in matched_det_indices:
                new_track = FeatureTrack(curr_features[j], global_track_id_counter)
                active_tracks.append(new_track)
                all_tracks.append(new_track)
                
                track_history[global_track_id_counter] = [global_idx]
                global_track_id_counter += 1
                
        # 5. Death: Handle Unmatched Tracks
        alive_tracks = []
        for i, track in enumerate(active_tracks):
            if i not in matched_track_indices:
                # Track was not matched. Does it exceed our patience?
                if track.time_since_update > max_missed_frames:
                    track.active = False
                else:
                    alive_tracks.append(track) # Kept alive in the dark
            else:
                alive_tracks.append(track)
                
        active_tracks = alive_tracks # Update the list for the next frame

    # 6. Post-processing: Filter noise and populate the final labels array
    for track in all_tracks:
        if track.hits >= min_hits:
            # Map the global Track ID to the original input array indices
            for original_idx in track_history[track.track_id]:
                labels[original_idx] = track.track_id
                
    # Cells that belonged to tracks with hits < min_hits remain as -1 (noise)
    return labels

In [ ]:
import napari
import tifffile
from tracking_functions import build_spatial_feature_graph_nx, cluster_spatial_graph, instance_mask_to_features, constrained_dbscan_tracking, dbscan_with_connected_components
import pickle as pkl
import numpy as np
from tqdm import tqdm

n_tested=50


# preds=pkl.load(open("./results/0.2-50.0/5_frames/3d_preds_v0_best.pkl", "rb"))
preds=pkl.load(open("./results/Fluo-N2DH-GOWT1/1_frames/preds_2d_seg.pkl", "rb"))
if len(preds[0][0].shape) == 4:
    print("3D predictions detected. Squeezing the channel dimension.")
    foreground_mask, dense_features = preds[0][0].squeeze(0) > 0.5, preds[0][1].squeeze(0)
    for i in range(1,n_tested):
        foreground_mask = np.concatenate((foreground_mask, preds[i][0].squeeze(0) > 0.5), axis=0)
        dense_features = np.concatenate((dense_features, preds[i][1].squeeze(0)), axis=0)

    print(foreground_mask.shape, dense_features.shape)
else:
    foreground_mask, dense_features = preds[0][0] > 0.5, preds[0][1]
    for i in range(1,n_tested):
        foreground_mask = np.concatenate((foreground_mask, preds[i][0] > 0.5), axis=0)
        dense_features = np.concatenate((dense_features, preds[i][1]), axis=0)

print("Shapes of foreground_mask and dense_features:", foreground_mask.shape, dense_features.shape)

#graph based segmentation
# instance_masks = np.zeros_like(foreground_mask, dtype=np.int32)
# for i in range(n_tested):
#     g_spatial, node_grid = build_spatial_feature_graph_nx(
#         dense_features[i], 
#         foreground_mask[i], 
#         distance_threshold=1.5)
#     print(f"Frame {i}: Graph has {g_spatial.number_of_nodes()} nodes and {g_spatial.number_of_edges()} edges.")
#     instance_mask = cluster_spatial_graph(
#         g_spatial, 
#         node_grid, 
#         foreground_mask[i],
#         min_cell_size=1000,
#         resolution=0.0000001
#     )
#     print(f"Frame {i}: Detected {len(np.unique(instance_mask)) - 1} instances.")
#     instance_masks[i] = instance_mask


#density based segmentation
instance_masks = np.zeros_like(foreground_mask, dtype=np.int32)
for i in range(n_tested):
    instance_mask = dbscan_with_connected_components(
        dense_features[i], 
        foreground_mask[i],
        eps=50,
        min_samples=10
    )
    print(f"Frame {i}: Detected {len(np.unique(instance_mask)) - 1} instances.")
    # print(f"Frame {i}: Unique instance IDs: {np.unique(instance_mask)}0")
    instance_masks[i] = instance_mask

instance_features=np.empty((0, dense_features.shape[1]), dtype=np.float32)
frames=[]
inst_locs=np.empty((0, 2), dtype=np.float32)
for i in range(n_tested):
    inst_features, inst_loc = instance_mask_to_features(instance_masks[i], dense_features[i])
    instance_features= np.concatenate((instance_features, inst_features), axis=0)
    frames= np.concatenate((frames, np.ones(inst_features.shape[0])*i), axis=0)
    inst_locs= np.concatenate((inst_locs, inst_loc), axis=0)
    
#kalman feature tracking
labels = kalman_feature_tracking(instance_features, frames, max_feature_distance=1.0, max_missed_frames=3, min_hits=3)

napari_tracks= np.concatenate([labels[:, np.newaxis],frames[:, np.newaxis], inst_locs.astype(np.int32)], axis=1).astype(np.int32)
napari_tracks_filtered=napari_tracks[napari_tracks[:,0]!=-1]
print(f"Found {len(np.unique(labels)) - 1} unique tracks (excluding noise).")
print(f"Track ID distribution: {np.unique(labels, return_counts=True)}")

viewer = napari.Viewer()
video_path="/home/tristan/data/CTC_SINETRA_seg/Fluo-N2DH-GOWT1/02/video.tiff"
viewer.add_image(tifffile.imread(video_path)[2:n_tested+2], name="video")
viewer.add_labels(instance_masks, name="instance masks")
viewer.add_tracks(napari_tracks_filtered)

In [ ]:
# visualize the instance features in 2D using Tsne
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=42)
instance_features_2d = tsne.fit_transform(instance_features)   
from matplotlib import pyplot as plt
plt.figure(figsize=(8, 6))
plt.scatter(instance_features_2d[:, 0], instance_features_2d[:, 1], c=frames, cmap='viridis', s=5)
plt.colorbar(label='Frame Index')
plt.title('t-SNE of Instance Features Colored by Frame Index')